# R1 — GA re-decision

R1 ไม่ train model ใหม่ แต่ให้ GA ค้นหากฎ MATCH/REVIEW/NO_MATCH จาก score เดิมของ R0.

In [ ]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import Markdown, display

def find_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents,
                  Path(r'D:/66070260-Year3_Term2/Project1/Code')]
    for candidate in candidates:
        if (candidate / 'exp_lib.py').exists(): return candidate
    raise FileNotFoundError('Project root containing exp_lib.py was not found')

ROOT = find_root(); EXP = ROOT / 'experiments'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

def source(module, *names):
    for name in names:
        display(Markdown(f'### `{module.__name__}.{name}`'))
        print(inspect.getsource(getattr(module, name)))

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

print('Project root:', ROOT)


## 1. Genome และ decision rule

Genome = `t_m, t_r, c_promote, c_demote`. Probability กำหนดช่วงหลัก และ `name_sim` ช่วยตัดสินคู่ก้ำกึ่ง.

In [ ]:
import exp_r1_ga_redecision as r1
source(r1, 'decide_code', 'cost_from_counts', 'run_ga')

## 2. Crossover และ mutation

GA สร้างประชากรของ rule, วัด validation cost, เก็บ elite, crossover และ mutate. Test ไม่ถูกใช้เลือก genome.

In [ ]:
source(r1, 'uniform_crossover', 'mutate')
r = read_json('experiments/r1_results.json')
display(pd.DataFrame(r['rules']['test']).T); print('Legacy genome:', r['best_genome'])

## 3. Nested automation result

การรันล่าสุดใช้หลาย GA seeds และเลือกจาก validation cost เท่านั้น.

In [ ]:
s = read_json('experiments/automation/r1_nested_primary_20260809/summary.json')
display(pd.DataFrame(s['trials'])); print(json.dumps(s['aggregates']['A_current'], ensure_ascii=False, indent=2))

In [ ]:
# Optional full rerun:
# import subprocess
# subprocess.run([sys.executable, str(ROOT/'run_ga_experiments.py'), '--experiment', 'r1', '--seeds', '7', '42', '123', '999', '2025'], check=True)